# **01 - SelfCheck-Ngram** 

In [ ]:
# Restart after running this cell.
!pip install -q "transformers==4.36.2" selfcheckgpt sentencepiece "accelerate>=0.26,<1.0" "bitsandbytes>=0.43.0"
!python -m spacy download en_core_web_sm -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.7.0 requires tokenizers>=0.19, but you have tokenizers 0.15.2 which is incompatible.
sentence-transformers 5.7.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to 

In [ ]:
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = "/content/drive/MyDrive/selfcheckgpt_replication_results"
except ImportError:                      # running outside Colab
    RESULTS_DIR = "./selfcheckgpt_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.environ["SC_RESULTS_DIR"] = RESULTS_DIR
if RESULTS_DIR not in sys.path: sys.path.insert(0, RESULTS_DIR)
try:
    from sc_common import *            # shared config, dataset, metrics, cache helpers 
except ModuleNotFoundError:
    raise FileNotFoundError(f"Upload sc_common.py into {RESULTS_DIR} first.")
describe_device(); ram("after setup")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Using device: cuda
Tesla T4 - 14.6 GB
Results / caches -> /content/drive/MyDrive/selfcheckgpt_replication_results
[RAM after setup] this process 0.7 GB | system used 1.3 / 12.7 GB


In [ ]:
from selfcheckgpt.modeling_selfcheck import SelfCheckNgram
nlp = get_nlp()

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [ ]:
t0 = time.time()
NGRAM_SCORES = {}
orders = [1, 2, 3, 4, 5] if RUN_NGRAM_ORDERS else [1]
for n in orders:
    scorer = SelfCheckNgram(n=n)
    def fn(i, scorer=scorer):
        r = rows[i]
        out = scorer.predict(sentences=r["gpt3_sentences"], passage=r["gpt3_text"], sampled_passages=get_samples(i, "samples"))
        return {"avg": np.array(out["sent_level"]["avg_neg_logprob"]), "max": np.array(out["sent_level"]["max_neg_logprob"]),
                "doc_avg": float(out["doc_level"]["avg_neg_logprob"]), "doc_max": float(out["doc_level"]["avg_max_neg_logprob"])}
    NGRAM_SCORES[(n, "samples")] = run_cached(f"ngram{n}_samples", fn, IDS, f"{n}-gram", save_every=20)
print(f"done in {(time.time() - t0) / 60:.1f} min")

SelfCheck-1gram initialized
1-gram: 0 cached, 238 to compute


1-gram:   0%|          | 0/238 [00:00<?, ?it/s]

SelfCheck-2gram initialized
2-gram: 0 cached, 238 to compute


2-gram:   0%|          | 0/238 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/selfcheckgpt/modeling_ngram.py:135: RuntimeWarning: divide by zero encountered in log
  logprob = np.log(train_prob)


SelfCheck-3gram initialized
3-gram: 0 cached, 238 to compute


... (2 outputs left out)

4-gram:   0%|          | 0/238 [00:00<?, ?it/s]

SelfCheck-5gram initialized
5-gram: 0 cached, 238 to compute


5-gram:   0%|          | 0/238 [00:00<?, ?it/s]

done in 32.7 min


In [ ]:
if RUN_WIKIBIO_EVIDENCE:      # n-gram model trained on the response + the WikiBio paragraph
    sc1 = SelfCheckNgram(n=1)
    def fn_wiki(i):
        r = rows[i]
        o = sc1.predict(sentences=r["gpt3_sentences"], passage=r["gpt3_text"], sampled_passages=get_samples(i, "wiki"))
        return {"avg": np.array(o["sent_level"]["avg_neg_logprob"]), "max": np.array(o["sent_level"]["max_neg_logprob"]),
                "doc_avg": float(o["doc_level"]["avg_neg_logprob"]), "doc_max": float(o["doc_level"]["avg_max_neg_logprob"])}
    NGRAM_SCORES[(1, "wiki")] = run_cached("ngram1_wiki", fn_wiki, IDS, "1-gram (WikiBio)", save_every=20)

SelfCheck-1gram initialized
1-gram (WikiBio): 0 cached, 238 to compute


1-gram (WikiBio):   0%|          | 0/238 [00:00<?, ?it/s]

In [ ]:
@functools.lru_cache(maxsize=None)
def _sent_tokens(text):
    return tuple(tuple(t.text for t in nlp(s.text.strip())) for s in nlp(text).sents)

@functools.lru_cache(maxsize=None)
def _tokens(sentence):
    return tuple(t.text for t in nlp(sentence))

def fast_unigram(sentences, passage, samples, lowercase=True):
    counts, total = Counter(), 0
    for text in [passage] + list(samples):
        for toks in _sent_tokens(text):
            toks = [t.lower() for t in toks] if lowercase else list(toks)
            counts.update(toks); total += len(toks)
    avg, mx, lp_doc = [], [], []
    with np.errstate(divide="ignore"):
        for sent in sentences:
            lp = [np.log(counts.get(t.lower() if lowercase else t, 0) / total) for t in _tokens(sent)]
            lp_doc += lp; avg.append(-np.mean(lp)); mx.append(-np.min(lp))
    return {"avg": np.array(avg), "max": np.array(mx), "doc_avg": float(-np.mean(lp_doc)), "doc_max": float(np.mean(mx))}


In [ ]:
if VERIFY_AGAINST_PACKAGE:
    for i in IDS[:3]:
        a, b = NGRAM_SCORES[(1, "samples")][i], fast_unigram(rows[i]["gpt3_sentences"], rows[i]["gpt3_text"], get_samples(i))
        check_close(f"fast unigram vs package, passage {i} (sentence Max)", a["max"], b["max"], 1e-8)
        check_close(f"fast unigram vs package, passage {i} (sentence Avg)", a["avg"], b["avg"], 1e-8)


[OK] fast unigram vs package, passage 0 (sentence Max): max |diff| = 0.00e+00 (tol 1e-08)
[OK] fast unigram vs package, passage 0 (sentence Avg): max |diff| = 0.00e+00 (tol 1e-08)
[OK] fast unigram vs package, passage 1 (sentence Max): max |diff| = 0.00e+00 (tol 1e-08)
[OK] fast unigram vs package, passage 1 (sentence Avg): max |diff| = 0.00e+00 (tol 1e-08)
[OK] fast unigram vs package, passage 2 (sentence Max): max |diff| = 0.00e+00 (tol 1e-08)
[OK] fast unigram vs package, passage 2 (sentence Avg): max |diff| = 0.00e+00 (tol 1e-08)


In [ ]:
# unigram (max) sentence scores for the first N samples, N in NS (cached)
if RUN_SAMPLE_ABLATION:
    def fn_abl(i):
        r = rows[i]
        return {n: fast_unigram(r["gpt3_sentences"], r["gpt3_text"], r["gpt3_text_samples"][:n])["max"] for n in NS}
    run_cached("unigram_ablation", fn_abl, IDS, "unigram ablation", save_every=20)
print("n-gram caches written to", RESULTS_DIR)

unigram ablation: 0 cached, 238 to compute


unigram ablation:   0%|          | 0/238 [00:00<?, ?it/s]

n-gram caches written to /content/drive/MyDrive/selfcheckgpt_replication_results
